# SFT 3단계 — 학습 효과 검증 (A/B 비교)

## 이 노트북이 하는 일
**같은 세션에서 베이스 모델과 RFT 모델을 둘 다 돌려** 같은 300문제로 비교합니다.

```
같은 300문제 (seed=42)
   ├─ 베이스 (어댑터 없음)   → maj@32, pass@32
   └─ RFT   (어댑터 적용)   → maj@32, pass@32
```

## 왜 같은 세션에서 둘 다 돌리나
어제 잰 베이스 점수(0.7433)를 그냥 써도 되지만, 그 사이 vLLM 버전이나 환경이 바뀌었을 수 있습니다.
**같은 조건에서 나란히 재는 게 가장 확실한 증거**입니다. `enable_lora=True`로 올리면 어댑터를 붙였다 뗐다 할 수 있어서 한 세션에서 가능합니다.

## 사전 준비 ⚠️
1. 학습 노트북에서 받은 `rft_lora.zip`을 **Kaggle Dataset으로 업로드** (Kaggle이 자동으로 압축을 풉니다)
2. 이 노트북에 **+ Add Input**으로 ① 대회 데이터 ② 그 어댑터 Dataset 둘 다 추가

## 기준선 (어제 측정값)
| 지표 | 베이스 |
|---|---|
| maj@8 | 0.7167 |
| maj@16 | 0.7367 |
| **maj@32** | **0.7433** |
| **pass@32** | **0.8633** |

**maj@32가 0.7433보다 오르면 성공.** pass@32는 거의 안 변할 것으로 예상됩니다(RFT는 천장을 못 올림).


---
## [1] 설정 ▶️

In [ ]:
N_SAMPLES  = 32        # 어제 maj@32=0.7433 과 직접 비교하기 위해 32 고정
TEMP       = 0.8       # 어제와 동일
MAX_TOKENS = 1024
VALID_N    = 300       # 절대 변경 금지 — 어제와 같은 300문제여야 비교 가능
SEED       = 42        # 절대 변경 금지
RUN_BASE   = True      # False로 두면 RFT만 측정 (시간 절반)
MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
LORA_RANK  = 32        # 학습 때 LORA_R과 같아야 함
print(f"{VALID_N}문제 x {N_SAMPLES}샘플 x {'2(base+rft)' if RUN_BASE else '1(rft)'}")

---
## [2] vLLM 설치 ⏭️ 세션 안 껐으면 건너뛰기

In [ ]:
!pip install -q -U vllm 2>&1 | tail -3

---
## [3] protobuf 고정 ⏭️ [2]와 함께

---
## ⛔ 여기서 Run → Restart Session → [1]부터 다시
---

In [ ]:
!pip install -q -U "protobuf>=6.33.6,<7" 2>&1 | tail -2
import google.protobuf as p
print("protobuf", p.__version__)
assert p.__version__.startswith("6.")

---
## [4] 데이터 + 어댑터 경로 찾기 ▶️

`adapter_config.json`이 있는 폴더를 자동으로 찾습니다.
Kaggle이 zip을 어떻게 풀었든(하위 폴더가 생기든) 알아서 잡습니다.

In [ ]:
import glob, os, pandas as pd

def find_csv(must_have, must_not=()):
    for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
        b = os.path.basename(p).lower()
        if all(k in b for k in must_have) and not any(k in b for k in must_not):
            return p
    return None

TRAIN_PATH = find_csv(["train"], must_not=["filtered","ids","leaderboard","test"])
BAD_PATH   = find_csv(["filtered","ids"])
assert TRAIN_PATH and TRAIN_PATH != BAD_PATH

train = pd.read_csv(TRAIN_PATH)
train = train[~train["id"].isin(set(pd.read_csv(BAD_PATH)["id"]))].reset_index(drop=True)
assert len(train) == 16373, f"16373이어야 하는데 {len(train)}"

# ⚠️ 어제와 완전히 동일한 300문제
work = train.sample(VALID_N, random_state=SEED).reset_index(drop=True)
gold = work["answer"].tolist()
print(f"검증셋 {len(work)}문제 | 첫 id: {work.iloc[0]['id']}")

# 어댑터 폴더 자동 탐색
cfgs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
assert cfgs, "adapter_config.json 을 못 찾았습니다. 어댑터 Dataset을 Input에 추가하세요."
LORA_PATH = os.path.dirname(cfgs[0])
print("어댑터:", LORA_PATH)
print(sorted(os.listdir(LORA_PATH)))

---
## [5] 답 추출기 ▶️ 어제와 완전히 동일

In [ ]:
import re
from collections import Counter

def extract_boxed(text):
    """Return the raw content inside the LAST \\boxed{...}, brace-balanced."""
    idx = text.rfind('\\boxed')
    if idx == -1:
        return None
    i = idx + len('\\boxed')
    while i < len(text) and text[i] == ' ':
        i += 1
    if i >= len(text):
        return None
    if text[i] != '{':                       # bare form: \boxed 15
        m = re.match(r'-?[\d,]+', text[i:])
        return m.group(0) if m else None
    depth, start = 0, i + 1
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i]
        i += 1
    return None

def to_int(s):
    """LaTeX/text -> python int, or None. Never uses float(), so huge ints survive."""
    if s is None:
        return None
    s = str(s).strip()
    s = s.replace('{,}', '').replace('{\\,}', '')          # LaTeX thousands separator
    s = re.sub(r'\\(?:text|mathrm|mbox|textbf|textrm)\s*\{([^{}]*)\}', r'\1', s)
    for junk in ['\\!', '\\,', '\\;', '\\:', '\\ ', '\\left', '\\right',
                 '\\$', '$', '%', '~', '^\\circ', '\\%']:
        s = s.replace(junk, '')
    s = s.replace(',', '').replace(' ', '').strip()
    s = re.sub(r'[a-zA-Z]+$', '', s)                       # trailing unit: 42cm -> 42
    while len(s) > 1 and s[0] == '(' and s[-1] == ')':     # (\frac{100}{4}) -> \frac{100}{4}
        s = s[1:-1].strip()
    s = s.rstrip('.')
    if not s:
        return None
    m = re.fullmatch(r'\\[dt]?frac\{([-+]?\d+)\}\{([-+]?\d+)\}', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)/([-+]?\d+)', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)(?:\\times|\\cdot)10\^\{?(\d+)\}?', s)
    if m:
        return int(m.group(1)) * 10 ** int(m.group(2))
    if re.fullmatch(r'[-+]?\d+', s):
        return int(s)
    m = re.fullmatch(r'([-+]?\d+)\.0*', s)
    if m:
        return int(m.group(1))
    return None

def last_int(text):
    for c in reversed(re.findall(r'-?\d[\d,]*', text)):
        v = to_int(c)
        if v is not None:
            return v
    return None

def parse_answer(text):
    """None means 'this sample produced no usable integer' -> dropped from voting."""
    raw = extract_boxed(text)
    if raw is not None:
        return to_int(raw)          # boxed present but unparseable -> None, do NOT guess
    m = re.findall(r'(?:answer|Answer|ANSWER)\s*(?:is|:|=)+\s*\$?(-?[\d,]+)', text)
    if m:
        v = to_int(m[-1])
        if v is not None:
            return v
    return last_int(text)

def majority_vote(values, fallback=0):
    vals = [v for v in values if v is not None]
    if not vals:
        return fallback
    return Counter(vals).most_common(1)[0][0]

_c = [(r"\boxed{132}",132), (r"\boxed{-2,025,078}",-2025078),
      (r"\boxed{\dfrac{650}{5}}",130), (r"\boxed{\frac{7}{2}}",None)]
print("parser FAILURES:", sum(parse_answer(t)!=w for t,w in _c), "/", len(_c))

---
## [6] 프롬프트 ▶️ 학습 때와 동일한 SYSTEM

학습 데이터를 이 프롬프트로 만들었으므로 **반드시 같아야** 합니다.
다르면 모델이 학습한 것과 다른 상황을 만나게 됩니다.

In [ ]:
from transformers import AutoTokenizer

SYSTEM = ("You are an expert competition mathematician. Solve the problem step by step, "
          "concisely. The final answer is ALWAYS a single integer. "
          "End your response with the final integer inside \\boxed{}.")

tok = AutoTokenizer.from_pretrained(MODEL_ID)
prompts = [tok.apply_chat_template(
    [{"role":"system","content":SYSTEM},{"role":"user","content":q}],
    tokenize=False, add_generation_prompt=True) for q in work["question"]]
print(len(prompts), "프롬프트 준비 완료")

---
## [7] 모델 로드 (LoRA 지원) ▶️

**`enable_lora=True`** — 어댑터를 붙였다 뗐다 할 수 있게 합니다.
`generate()`에 `lora_request`를 주면 어댑터 적용, 안 주면 베이스 그대로.

**`max_lora_rank`** — 학습 때 쓴 `r` 이상이어야 합니다. 우리는 32.

In [ ]:
import time, numpy as np
from collections import Counter
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(model=MODEL_ID, dtype="half", max_model_len=4096,
          gpu_memory_utilization=0.90, tensor_parallel_size=1,
          seed=SEED, trust_remote_code=True,
          enable_lora=True, max_lora_rank=LORA_RANK)

sp = SamplingParams(n=N_SAMPLES, temperature=TEMP, top_p=0.95,
                    max_tokens=MAX_TOKENS, seed=SEED)
print("로드 완료")

---
## [8] 채점 함수 ▶️ (GPU 미사용)

`maj@k`(다수결 정확도)와 `pass@k`(정답이 후보에 존재하는 비율)를 같이 계산합니다.

- **`maj@k`가 오르면** → RFT 성공. 정답 경로의 확률이 올라간 것
- **`pass@k`도 오르면** → 기대 이상. 없던 능력이 생긴 것
- **`pass@k`가 떨어지면** ⚠️ 과적합. 다양성이 줄어 탐색 범위가 좁아진 것

In [ ]:
def score(outs, gold, label):
    maj, psk, shares, fails = [], [], [], 0
    for o, g in zip(outs, gold):
        vals  = [parse_answer(c.text) for c in o.outputs]
        fails += sum(v is None for v in vals)
        valid = [v for v in vals if v is not None]
        cnt   = Counter(valid)
        m     = cnt.most_common(1)[0][0] if cnt else 0
        maj.append(int(m) == int(g))
        psk.append(any(int(v) == int(g) for v in valid))
        shares.append(cnt[m]/len(vals) if cnt else 0)
    r = {"label": label, "maj": np.mean(maj), "pass": np.mean(psk),
         "share": np.mean(shares), "fail": fails/(len(outs)*N_SAMPLES),
         "maj_list": maj}
    print(f"[{label}]  maj@{N_SAMPLES}={r['maj']:.4f}  pass@{N_SAMPLES}={r['pass']:.4f}  "
          f"평균득표율={r['share']:.3f}  파싱실패={r['fail']:.2%}")
    return r

results = {}

---
## [9] 베이스 모델 측정 ▶️ 약 26분

`lora_request`를 **주지 않으면** 어댑터 없는 원본 모델로 생성됩니다.
어제 값(0.7433)과 비슷하게 나오면 환경이 동일하다는 확인이 됩니다.

In [ ]:
if RUN_BASE:
    t0 = time.time()
    outs_base = llm.generate(prompts, sp)          # lora_request 없음 = 베이스
    print(f"생성 {(time.time()-t0)/60:.1f}분")
    results["base"] = score(outs_base, gold, "BASE")
else:
    print("건너뜀 (RUN_BASE=False)")

---
## [10] RFT 모델 측정 ▶️ 약 26분

`lora_request`를 주면 어댑터가 적용됩니다.

In [ ]:
t0 = time.time()
outs_rft = llm.generate(prompts, sp,
                        lora_request=LoRARequest("rft", 1, LORA_PATH))
print(f"생성 {(time.time()-t0)/60:.1f}분")
results["rft"] = score(outs_rft, gold, "RFT")

---
## [11] 결론 ▶️

### 대응 비교(paired)라 민감합니다
같은 300문제를 두 모델이 각각 풀었으므로, **문제별로 짝지어 비교**할 수 있습니다.
"베이스는 틀렸는데 RFT는 맞힌 문제"와 그 반대를 세면, 단순 정확도 차이보다 훨씬 확실한 판단이 됩니다.

### 판정 기준
- **개선 +3%p 이상** → 채택. 리더보드 제출로 확인
- **+1~3%p** → 애매. 아래 McNemar 수치를 보고 판단
- **하락** → 학습 실패. `EPOCHS=1` 또는 `LR=5e-5`로 재시도

In [ ]:
if "base" in results:
    b, r = results["base"], results["rft"]
    print(f"maj@{N_SAMPLES} :  {b['maj']:.4f}  →  {r['maj']:.4f}   ({(r['maj']-b['maj'])*100:+.2f}%p)")
    print(f"pass@{N_SAMPLES}:  {b['pass']:.4f}  →  {r['pass']:.4f}   ({(r['pass']-b['pass'])*100:+.2f}%p)")
    print(f"평균 득표율:  {b['share']:.3f}  →  {r['share']:.3f}   ({r['share']-b['share']:+.3f})")

    win  = sum((not x) and y for x, y in zip(b["maj_list"], r["maj_list"]))
    lose = sum(x and (not y) for x, y in zip(b["maj_list"], r["maj_list"]))
    print(f"\nRFT만 맞힌 문제: {win}개")
    print(f"베이스만 맞힌 문제: {lose}개")
    print(f"→ 순증: {win-lose:+d}개")
    if win + lose >= 10:
        import math
        z = abs(win - lose) / math.sqrt(win + lose)
        print(f"→ McNemar z = {z:.2f}  ({'유의미' if z > 1.96 else '노이즈 범위'})")

    print("\n※ 평균 득표율이 크게 올랐다면 RFT가 의도대로 작동한 것입니다")
    print("  (정답 경로의 확률이 올라가 샘플들이 한 답으로 모임)")
else:
    r = results["rft"]
    print(f"RFT maj@{N_SAMPLES} = {r['maj']:.4f}   (어제 베이스 기준선 0.7433)")
    print(f"RFT pass@{N_SAMPLES} = {r['pass']:.4f}   (어제 베이스 기준선 0.8633)")